In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
df = pd.read_csv("../data/raw/13-recommendation-systems-adult-census-income.csv")
df

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K


In [4]:
df.info(), df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


(None, (32561, 15))

In [5]:
df.select_dtypes(include="int").describe().T

,count,mean,std,min,25%,50%,75%,max
age,32561.0,38.581647,13.640433,17.0,28.0,37.0,48.0,90.0
fnlwgt,32561.0,189778.366512,105549.977697,12285.0,117827.0,178356.0,237051.0,1484705.0
education.num,32561.0,10.080679,2.572720,1.0,9.0,10.0,12.0,16.0
capital.gain,32561.0,1077.648844,7385.292085,0.0,0.0,0.0,0.0,99999.0
capital.loss,32561.0,87.303830,402.960219,0.0,0.0,0.0,0.0,4356.0
hours.per.week,32561.0,40.437456,12.347429,1.0,40.0,40.0,45.0,99.0


In [6]:
df.isnull().sum()
df = df.drop_duplicates()
df.columns


Index(['age', 'workclass', 'fnlwgt', 'education', 'education.num',
       'marital.status', 'occupation', 'relationship', 'race', 'sex',
       'capital.gain', 'capital.loss', 'hours.per.week', 'native.country',
       'income'],
      dtype='object')

In [7]:
df = df.drop(["fnlwgt","education","relationship","race","native.country"],axis=1)
df.shape

(32537, 10)

In [8]:
df = df.replace("?",np.nan)
df = df.dropna()
df["income"].apply(lambda x: 1 if x.strip() == ">50K" else 0)

1        0
3        0
4        0
5        0
6        0
        ..
32556    0
32557    0
32558    1
32559    0
32560    0
Name: income, Length: 30694, dtype: int64

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30694 entries, 1 to 32560
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             30694 non-null  int64 
 1   workclass       30694 non-null  object
 2   education.num   30694 non-null  int64 
 3   marital.status  30694 non-null  object
 4   occupation      30694 non-null  object
 5   sex             30694 non-null  object
 6   capital.gain    30694 non-null  int64 
 7   capital.loss    30694 non-null  int64 
 8   hours.per.week  30694 non-null  int64 
 9   income          30694 non-null  object
dtypes: int64(5), object(5)
memory usage: 2.6+ MB


In [10]:
X = df.drop("income", axis=1)
y = df["income"]

### Separar columnas categoricas y numericas
-Las columnas categóricas y numéricas necesitan transformaciones diferentes(codificado y escalado).

In [11]:
categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

X_cat = X[categorical_cols]
X_num = X[numerical_cols]

### Codificado

In [12]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_cat_encoded = encoder.fit_transform(X_cat)
X_cat_encoded

array([[0., 0., 1., ..., 0., 1., 0.],
       [0., 0., 1., ..., 0., 1., 0.],
       [0., 0., 1., ..., 0., 1., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 1.],
       [0., 0., 1., ..., 0., 1., 0.],
       [0., 0., 1., ..., 0., 0., 1.]], shape=(30694, 30))

### Escalado

In [13]:
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)
X_num_scaled

array([[ 3.32082234, -0.4418007 , -0.14757495, 10.5150421 , -1.91525627],
       [ 1.18585272, -2.39398176, -0.14757495,  9.3913401 , -0.07950149],
       [ 0.19461682, -0.05136448, -0.14757495,  9.3913401 , -0.07950149],
       ...,
       [ 0.11836791, -0.4418007 , -0.14757495, -0.21926909, -0.07950149],
       [ 1.49084838, -0.4418007 , -0.14757495, -0.21926909, -0.07950149],
       [-1.25411256, -0.4418007 , -0.14757495, -0.21926909, -1.74836947]],
      shape=(30694, 5))

### Unir en matrices finales

In [14]:
X_final = np.concatenate([X_num_scaled, X_cat_encoded], axis=1)
X_final

array([[ 3.32082234, -0.4418007 , -0.14757495, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.18585272, -2.39398176, -0.14757495, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.19461682, -0.05136448, -0.14757495, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [ 0.11836791, -0.4418007 , -0.14757495, ...,  0.        ,
         0.        ,  1.        ],
       [ 1.49084838, -0.4418007 , -0.14757495, ...,  0.        ,
         1.        ,  0.        ],
       [-1.25411256, -0.4418007 , -0.14757495, ...,  0.        ,
         0.        ,  1.        ]], shape=(30694, 35))

### Matriz de similitud
-Encuentra qué usuarios tienen perfiles parecidos segun las features

### HAY QUE APLICAR KNN EN VEZ DE LA MATRIZ DE SIMILITUD O SIMILITUDES DE UN SUAURIO EN CONCRETO(VER VIDEO OTRA VEZ Y PREGUNTAR AL PROFESOR)

In [19]:
similitary_matrix = cosine_similarity(X_final)
similitary_matrix

: 

### Funcion de recomendación de similitud

In [15]:
def trayectoria(usuario_idx, df_original, X, y, top_k=5):
    #Similitud del usuario con los demás usuarios
    '''Vector con valores flotantes entre 0 y 1, que representa las similitudes entre usuarios, 
    mientras mas cercano al 1 el usuario se parece mas al usuario actual.'''
    similitud = similitary_matrix[usuario_idx]

    #Lista de usuarios ordenados
    '''argsort():Devuelve los indices ordenados de menor a mayor
    [::-1]:Invierte el orden(de mayor a menor)
    [1:]:Quita el primer elemento(usuario actual)'''
    similar_users = similitud.argsort()[::-1][1:]

    #Filtramos por ingresos altos
    '''y.iloc[i] == 1
    -income = 1 -> gana mas de 50k
    -income = 0 -> gana menos de 50k
    -[:top_k] -> tomamos solo los primeros 5
    Nos quedamos solo con los usuarios PARECIDOS que ganan más'''
    ganan_mas_50k = [i for i in similar_users if y.iloc[i] == 1][:top_k]
    ganan_mas_50k
    
    #DataFrame con caracteristicas relevantes
    '''Trayectoria que siguieron personas parecidas que lograron mejores ingresos'''

    return df_original.iloc[ganan_mas_50k][['education.num', 'occupation', 'hours.per.week']]



In [16]:
#Usuario de prueba
'''-Filtramos el df y nos quedamos solo con los usuarios que ganan menos de 50k
-tomamos los indices de esos usuarios
-selecciona el primer usuario de ese grupo'''
user_test = df[df['income'] == 0].index[0]

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
trayectoria(user_test, df, X, y)